# Week 2: Learn Quantum Mechanics Using a Quantum Computer
## Quantum Coins — Superposition and Interference

**Module:** [Superposition with Qiskit](https://quantum.cloud.ibm.com/learning/en/modules/quantum-mechanics/superposition-with-qiskit)

In this notebook, we explore **superposition** — one of the core principles of quantum theory — by comparing classical coin flips with quantum "coin flips" using Qiskit. Along the way, we'll discover:

1. How a **Hadamard gate** creates a superposition analogous to a coin flip
2. Why a quantum superposition is fundamentally **different** from a classical probability distribution — thanks to **phase** and **interference**
3. How the **Bloch sphere** provides an intuitive 3D picture of a qubit's state

> **Note:** We use `AerSimulator` for all experiments so no IBM Quantum account or QPU time is required.

---
## SECTION 1: Setup & Imports

In [ ]:
# Core Qiskit
from qiskit import QuantumCircuit
from qiskit.quantum_info import Pauli
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Aer simulator (local, no IBM account needed)
from qiskit_aer import AerSimulator

# Primitives (local backend versions)
from qiskit.primitives import BackendSamplerV2, BackendEstimatorV2

# Standard libraries
import numpy as np
import matplotlib.pyplot as plt
import random

# Create the simulator backend
backend = AerSimulator()
print(f"Using backend: {backend.name}")

---
## SECTION 2: Introduction — Classical Coin vs Quantum Coin

In everyday life, objects have definite properties. In the quantum world, a quantum object can be in a **superposition** of multiple classically allowed states. When measured, the superposition randomly "collapses" to one of those states.

We'll use a classical coin flip as an analogy — and then show where the analogy breaks down.

### 2.1 Classical Coin Flip

A fair coin has a 50/50 chance of landing heads-up or heads-down. We write its state as a **classical probabilistic state**:

$$
S(\text{coin}) = \frac{1}{2} |\text{up}\rangle + \frac{1}{2} |\text{down}\rangle
$$

The coefficients here are the **probabilities** themselves. Let's simulate 1000 classical coin flips:

In [ ]:
# Classical coin flip simulation
nflips = 1000
fliplist = [random.randint(0, 1) for _ in range(nflips)]

heads_up = fliplist.count(0)
heads_down = fliplist.count(1)
print(f"Heads Up (0): {heads_up}, Heads Down (1): {heads_down}")

plt.hist(fliplist, bins=2, edgecolor='black', rwidth=0.8)
plt.xticks([0.25, 0.75], ['Heads Up (0)', 'Heads Down (1)'])
plt.ylabel('Counts')
plt.title('Classical Coin Flip — 1000 flips')
plt.show()

As expected, we get roughly 500 heads-up and 500 heads-down. Now let's see if a quantum "coin" behaves the same way.

### 2.2 Quantum Coin — The Hadamard Gate

A qubit can be measured in two states: $|0\rangle$ and $|1\rangle$. Starting from $|0\rangle$, we apply a **Hadamard gate** to create an equal superposition:

$$
|\psi\rangle = \frac{1}{\sqrt{2}} |0\rangle + \frac{1}{\sqrt{2}} |1\rangle
$$

Unlike the classical case, the coefficients are **amplitudes** — the **square** of these amplitudes gives the probabilities:

$$
P(0) = \left|\frac{1}{\sqrt{2}}\right|^2 = \frac{1}{2}, \quad P(1) = \left|\frac{1}{\sqrt{2}}\right|^2 = \frac{1}{2}
$$

In [ ]:
# Build the quantum coin circuit: H gate + measurement
qcoin = QuantumCircuit(1)
qcoin.h(0)
qcoin.measure_all()

qcoin.draw('mpl')

In [ ]:
# Transpile and run with Sampler (1000 shots)
pm = generate_preset_pass_manager(target=backend.target, optimization_level=3)
qc_isa = pm.run(qcoin)

sampler = BackendSamplerV2(backend=backend)
job = sampler.run([qc_isa], shots=1000)
res = job.result()

counts = res[0].data.meas.get_counts()
print(f"Quantum coin results: {counts}")

plot_histogram(counts)

The quantum coin histogram looks essentially identical to the classical coin histogram — roughly 50/50 split between 0 and 1. So far, the two coins look the same!

### 2.3 Expectation Values with Estimator

In addition to sampling, we can measure the **expectation value** of an observable. Using the **Pauli-Z** operator:
- $Z|0\rangle = +1 \cdot |0\rangle$  
- $Z|1\rangle = -1 \cdot |1\rangle$

For our 50/50 superposition:

$$
\langle \psi | Z | \psi \rangle = \frac{1}{2}(+1) + \frac{1}{2}(-1) = 0
$$

This is like a gambling game where you win \$1 for heads-up and lose \$1 for heads-down — the expected payoff is \$0.

In [ ]:
# Build circuit WITHOUT measurement (Estimator needs statevector)
qcoin_est = QuantumCircuit(1)
qcoin_est.h(0)

obs_Z = Pauli("Z")

# Transpile
qc_isa = pm.run(qcoin_est)
obs_Z_isa = obs_Z.apply_layout(layout=qc_isa.layout)

# Run Estimator
estimator = BackendEstimatorV2(backend=backend)
job = estimator.run([[qc_isa, obs_Z_isa]])
res = job.result()

print(f"<Z> expectation value: {res[0].data.evs:.4f}")
print("Expected: 0 (confirming 50/50 probability)")

---
## SECTION 3: The Quantum Revealed — Measuring Along Different Axes

Here's where the classical analogy breaks down.

**Thought experiment:** Flip a classical coin, but instead of letting it land flat, catch it between your palms so it's standing sideways. The probability of heads-left or heads-right is still 50/50 — the measurement axis doesn't matter for a classical coin.

But what about the quantum coin? We can measure along the X axis using the Pauli-X observable:
- $X|+\rangle = +1 \cdot |+\rangle$
- $X|-\rangle = -1 \cdot |-\rangle$

If it behaved classically, we'd expect $\langle X \rangle = 0$.

In [ ]:
# Measure the quantum coin along the X axis
qcoin_lr = QuantumCircuit(1)
qcoin_lr.h(0)

obs_X = Pauli("X")

# Transpile
qc_isa = pm.run(qcoin_lr)
obs_X_isa = obs_X.apply_layout(layout=qc_isa.layout)

# Run Estimator
estimator = BackendEstimatorV2(backend=backend)
job = estimator.run([[qc_isa, obs_X_isa]])
res = job.result()

print(f"<X> expectation value: {res[0].data.evs:.4f}")
print("\nSurprise! <X> ≈ 1, not 0!")
print("This means the state is ALWAYS |+⟩ when measured along X.")
print("A classical coin can't do this — random in one direction but certain in another!")

**Key Insight:** The quantum coin is random when measured along Z (50/50 for $|0\rangle$ vs $|1\rangle$), but perfectly determined when measured along X (always $|+\rangle$). No classical probabilistic system can behave this way!

---
## SECTION 4: Quantum Phase and Interference

### 4.1 What is Phase?

In a classical state, the coefficients are just positive real numbers (probabilities). In a quantum state:

$$
|\psi\rangle = c_1 |0\rangle + c_2 |1\rangle
$$

the coefficients $c_i$ are **complex numbers**: $c_i = |c_i| e^{i\phi_i}$

The **phase** $\phi_i$ determines how terms **interfere** — constructively (in phase) or destructively (out of phase).

### 4.2 Double Hadamard — Interference in Action

What happens if we apply the Hadamard gate **twice**?

$$
H|0\rangle = \frac{1}{\sqrt{2}} |0\rangle + \frac{1}{\sqrt{2}} |1\rangle
$$

$$
H|1\rangle = \frac{1}{\sqrt{2}} |0\rangle - \frac{1}{\sqrt{2}} |1\rangle
$$

Applying the second H to the superposition:

$$
H\left(\frac{1}{\sqrt{2}} |0\rangle + \frac{1}{\sqrt{2}} |1\rangle\right) = \frac{1}{2}[(|0\rangle + |1\rangle) + (|0\rangle - |1\rangle)] = |0\rangle
$$

The $|0\rangle$ terms interfere **constructively**, while the $|1\rangle$ terms interfere **destructively** and cancel!

In [ ]:
# Double Hadamard circuit
qcoin_hh = QuantumCircuit(1)
qcoin_hh.h(0)
qcoin_hh.h(0)
qcoin_hh.measure_all()

qcoin_hh.draw('mpl')

In [ ]:
# Run the double Hadamard
qc_isa = pm.run(qcoin_hh)

sampler = BackendSamplerV2(backend=backend)
job = sampler.run([qc_isa], shots=1000)
res = job.result()

counts = res[0].data.meas.get_counts()
print(f"Double Hadamard results: {counts}")
print("The second H cancels the first — we're back to |0⟩!")

plot_histogram(counts)

### 4.3 Adding a Phase Shift

Now let's insert a **phase gate** ($P(\pi)$) between the two Hadamards. This adds a phase of $\pi$ to the $|1\rangle$ component, flipping the sign:

$$
P(\pi)|1\rangle = e^{i\pi}|1\rangle = -|1\rangle
$$

This reverses which terms interfere constructively vs destructively!

In [ ]:
# H - Phase(π) - H circuit
qcoin_pi = QuantumCircuit(1)
qcoin_pi.h(0)
qcoin_pi.p(np.pi, 0)
qcoin_pi.h(0)
qcoin_pi.measure_all()

qcoin_pi.draw('mpl')

In [ ]:
# Run H-P(π)-H
qc_isa = pm.run(qcoin_pi)

sampler = BackendSamplerV2(backend=backend)
job = sampler.run([qc_isa], shots=1000)
res = job.result()

counts = res[0].data.meas.get_counts()
print(f"H-P(π)-H results: {counts}")
print("Now we always get |1⟩ — the phase completely reversed the outcome!")

plot_histogram(counts)

### 4.4 Exercise: Finding a Specific Phase

**Task:** Find the phase $\phi$ such that after H → $R_z(\phi)$ → H, the probabilities are:
- $P(|0\rangle) = 75\%$
- $P(|1\rangle) = 25\%$

**Hint:** Starting from $|0\rangle$:

$$
H \cdot R_z(\phi) \cdot H |0\rangle \Rightarrow P(|1\rangle) = \sin^2\left(\frac{\phi}{2}\right)
$$

So we need $\sin^2(\phi/2) = 0.25$, i.e., $\sin(\phi/2) = 0.5$, i.e., $\phi = \pi/3$.

In [ ]:
# Exercise: set the phase to get 75/25 split
phi = np.pi / 3  # <-- Solution: π/3 gives sin²(π/6) = 0.25 for |1⟩

qcoin_phase = QuantumCircuit(1)
qcoin_phase.h(0)
qcoin_phase.rz(phi, 0)
qcoin_phase.h(0)
qcoin_phase.measure_all()

# Transpile and run
qc_isa = pm.run(qcoin_phase)

sampler = BackendSamplerV2(backend=backend)
job = sampler.run([qc_isa], shots=4000)
res = job.result()

counts = res[0].data.meas.get_counts()
print(f"Phase exercise results: {counts}")
print(f"Expected: ~75% |0⟩, ~25% |1⟩")

plot_histogram(counts)

---
## SECTION 5: The $\sqrt{\text{NOT}}$ Gate — A Better Coin Analogy

The Hadamard isn't really like flipping a coin. A better analogy: imagine a coin sitting flat (heads up) on a table. Flipping it over (NOT gate) is a 180° rotation. A **$\sqrt{\text{NOT}}$** is a 90° rotation — the coin ends up on its edge!

When measured:
- **Along Z (up/down):** random, 50/50 → like squashing the coin flat
- **Along X (left/right):** random, 50/50
- **Along Y (forward/back):** deterministic! Always pointing one way

In Qiskit, the $\sqrt{\text{NOT}}$ gate is `sx()`.

In [ ]:
# Build √NOT circuit
qcoin_sx = QuantumCircuit(1)
qcoin_sx.sx(0)

qcoin_sx.draw('mpl')

In [ ]:
# Measure expectation values along X, Y, Z
obs_X = Pauli("X")
obs_Y = Pauli("Y")
obs_Z = Pauli("Z")

# Transpile
qc_isa = pm.run(qcoin_sx)
obs_X_isa = obs_X.apply_layout(layout=qc_isa.layout)
obs_Y_isa = obs_Y.apply_layout(layout=qc_isa.layout)
obs_Z_isa = obs_Z.apply_layout(layout=qc_isa.layout)

# Run Estimator with all three observables
estimator = BackendEstimatorV2(backend=backend)
pubs = [(qc_isa, [[obs_X_isa], [obs_Y_isa], [obs_Z_isa]])]
job = estimator.run(pubs)
res = job.result()

evs = res[0].data.evs.flatten()
print(f"<X> = {evs[0]:.4f}  (expect ≈ 0: random along X)")
print(f"<Y> = {evs[1]:.4f}  (expect ≈ -1: deterministic along Y)")
print(f"<Z> = {evs[2]:.4f}  (expect ≈ 0: random along Z)")

**Result:** X and Z expectation values are ≈ 0 (random, 50/50), but Y ≈ −1 (deterministic). This matches the coin-on-its-edge analogy perfectly!

> A coin **sitting still on its edge** is a much better visualization of a superposition state than a coin wildly flipping through the air.

---
## SECTION 6: The Bloch Sphere

An arbitrary single-qubit state can be written as:

$$
|\psi\rangle = \cos\frac{\theta}{2} |0\rangle + e^{i\phi} \sin\frac{\theta}{2} |1\rangle
$$

This maps to a point on the **Bloch sphere** — a unit sphere where:
- $|0\rangle$ is at the north pole
- $|1\rangle$ is at the south pole
- Superpositions lie on the equator or between the poles
- The phase $\phi$ determines the azimuthal angle

All single-qubit gates are **rotations** of this Bloch vector.

### 6.1 Visualizing Gates on the Bloch Sphere

Let's see how each gate rotates the Bloch vector starting from $|0\rangle$:

| Gate | Rotation | Bloch vector result |
|------|----------|--------------------|
| NOT (X) | 180° around x-axis | South pole ($|1\rangle$) |
| $\sqrt{\text{NOT}}$ (SX) | 90° around x-axis | Equator, pointing along $-y$ |
| PHASE($\pi$) | $\pi$ around z-axis | North pole (global phase only) |
| Hadamard | 90° around y, then 180° around x | Equator, pointing along $+x$ |

In [ ]:
# NOT gate
qnot = QuantumCircuit(1)
qnot.x(0)

print("NOT gate (X): rotates |0⟩ to |1⟩ (south pole)")
plot_bloch_multivector(qnot)

In [ ]:
# √NOT gate
qsqrtnot = QuantumCircuit(1)
qsqrtnot.sx(0)

print("√NOT gate (SX): rotates |0⟩ to equator, along -y")
plot_bloch_multivector(qsqrtnot)

In [ ]:
# Phase gate P(π)
qphase = QuantumCircuit(1)
qphase.p(np.pi, 0)

print("Phase gate P(π): only adds global phase to |0⟩ (stays at north pole)")
plot_bloch_multivector(qphase)

In [ ]:
# Hadamard gate
qhadamard = QuantumCircuit(1)
qhadamard.h(0)

print("Hadamard gate (H): rotates |0⟩ to equator, along +x")
plot_bloch_multivector(qhadamard)

---
## SECTION 7: Challenge Problem

**Task:** Create a circuit that transforms $|0\rangle$ to the state:

$$
|\psi\rangle = \frac{\sqrt{3}}{2}|0\rangle + \frac{1}{2} e^{i \frac{5\pi}{6}} |1\rangle
$$

**Approach:** Using the Bloch sphere parameterization $|\psi\rangle = \cos\frac{\theta}{2}|0\rangle + e^{i\phi}\sin\frac{\theta}{2}|1\rangle$:
- $\cos(\theta/2) = \sqrt{3}/2 \Rightarrow \theta/2 = \pi/6 \Rightarrow \theta = \pi/3$
- $\phi = 5\pi/6$

We can use $R_y(\theta)$ to set the polar angle, then $R_z(\phi)$ to set the azimuthal angle (phase).

In [ ]:
# Challenge solution
theta = np.pi / 3
phi = 5 * np.pi / 6

qc_challenge = QuantumCircuit(1)
qc_challenge.ry(theta, 0)   # Set polar angle θ
qc_challenge.rz(phi, 0)     # Set azimuthal angle φ

print("Challenge circuit:")
display(qc_challenge.draw('mpl'))

print("\nBloch sphere visualization:")
plot_bloch_multivector(qc_challenge)

In [ ]:
# Verify the statevector
from qiskit.quantum_info import Statevector

sv = Statevector.from_instruction(qc_challenge)
print(f"Statevector: {sv}")
print(f"\nExpected amplitudes:")
print(f"  c₀ = √3/2 = {np.sqrt(3)/2:.4f}")
print(f"  c₁ = (1/2)·e^(i·5π/6) = {0.5 * np.exp(1j * 5*np.pi/6):.4f}")
print(f"\nProbabilities: P(0)={abs(sv[0])**2:.4f}, P(1)={abs(sv[1])**2:.4f}")
print(f"Expected:      P(0)=0.7500, P(1)=0.2500")

---
## SECTION 8: Summary & Key Takeaways

### Critical Concepts:

1. **Superposition ≠ Classical Randomness:** While measuring a qubit in superposition gives random outcomes like a coin flip, the superposition state itself has **phase coherence** that allows constructive and destructive interference.

2. **Phase Matters:** The quantum phase determines how amplitudes interfere. The same gate (e.g., Hadamard) can have opposite effects depending on the phase of the input state.

3. **Bloch Sphere:** A single isolated qubit's state is a point on the Bloch sphere. The polar angle $\theta$ determines the amplitudes, and the azimuthal angle $\phi$ determines the relative phase.

4. **Gates = Rotations:** All single-qubit gates are rotations of the Bloch vector. They are deterministic and reversible. Randomness only enters at **measurement**.

5. **Better Analogy:** A coin sitting still on its edge (freely oriented in 3D) is a much better analogy for a superposition state than a coin flipping through the air.

### Questions for Reflection:

| # | Question |
|---|----------|
| 1 | **T/F:** A quantum superposition is basically the same as a probabilistic event in classical physics. |
| 2 | **T/F:** The length of the Bloch vector for a single isolated qubit is always 1. |
| 3 | **T/F:** Single-qubit quantum gates do not change the length of the Bloch vector. |
| 4 | Why can the state of a qubit be visualized on the Bloch sphere, but the probability distribution of a coin flip cannot? |
| 5 | Why is a coin flipping in the air not the best analogy to a quantum superposition state? |

### Answers:

1. **False.** Classical randomness lacks phase coherence — there's no interference. A superposition can interfere constructively or destructively, which a classical probability distribution cannot.

2. **True.** An isolated (pure) qubit state always lies on the surface of the Bloch sphere (radius = 1).

3. **True.** Unitary gates are norm-preserving rotations; they rotate the Bloch vector without changing its length.

4. A qubit state has both amplitude and phase, requiring three parameters $(r, \theta, \phi)$ — it naturally maps onto a sphere. A classical coin flip is described by a single real probability $p$, which is just a point on a line segment $[0, 1]$.

5. A flipping coin suggests uncontrolled randomness, but a superposition state is a **deterministic, coherent** state. The randomness only appears upon measurement. A coin sitting on its edge better captures the idea: it's in a definite orientation in 3D space, and the randomness only enters when you "collapse" it by squashing it flat along a measurement axis.